# Generate trips.txt

Generates GTFS trips file linking routes, services, and shapes.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import geopandas as gpd
from gtfs_common import route_id_from_trip_id


## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]
AGENCY_NAME = p["agency"]["name"]
AGENCY_ID = p["agency"]["id"]
AGENCY_URL = p["agency"]["url"]
AGENCY_TIMEZONE = p["agency"]["timezone"]
AGENCY_LANG = p["agency"]["lang"]
START_DATE = p["calendar"]["start_date"]
END_DATE = p["calendar"]["end_date"]
SERVICE_ID = p["calendar"]["SERVICE_ID"]

## Parámetros

In [6]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CITY}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read files

In [7]:
stop_times_df = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")
stop_times_df.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0000,1,00:00:00,00:00:12
1,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0001,2,00:00:47,00:00:59
2,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0002,3,00:01:34,00:01:46
3,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0003,4,00:02:22,00:02:34
4,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0004,5,00:03:09,00:03:21


In [8]:
routes_gdf = gpd.read_file(PATH_DIR_proccesed / "routes_clean.geojson")
routes_gdf.head(2)

,route_name,route_name_short,route_type,agency_id,shape_id,geometry
0,2_42 Caseta,42 Caseta,3,Red_de_Transporte_Merida,Shape_2_42 Caseta,"LINESTRING (1478265.739 2346761.139, 1478081.3..."
1,6_42 Sur Imss,42 Sur Imss,3,Red_de_Transporte_Merida,Shape_6_42 Sur Imss,"LINESTRING (1478238.108 2346765.115, 1478079.2..."


## Construcción de trips.txt

A partir de los viajes únicos en `stop_times_df`, se asigna un único `service_id`, se incorpora `shape_id` desde `routes_gdf` y se arma la tabla GTFS: route_id, service_id, trip_id, trip_headsign, direction_id, shape_id.


In [ ]:
stop_times_df["route_id"] = stop_times_df["trip_id"].map(route_id_from_trip_id)
stop_times_df[["trip_id", "route_id"]].drop_duplicates().head()


In [ ]:
trips_base = (
    stop_times_df[["trip_id", "route_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

routes_shape = routes_gdf[["route_id", "shape_id"]].drop_duplicates()
trips_base = trips_base.merge(routes_shape, on="route_id", how="left")
if trips_base["shape_id"].isna().any():
    missing = trips_base.loc[trips_base["shape_id"].isna(), "route_id"].tolist()
    raise ValueError(f"No shape_id for routes: {missing}")

trips_base["service_id"] = SERVICE_ID
trips_base["trip_headsign"] = ""
trips_base["direction_id"] = 1

cols_trips = ["route_id", "service_id", "trip_id", "trip_headsign", "direction_id", "shape_id"]
trips_df = trips_base[cols_trips]
trips_df.head(10)


## Export

In [11]:
trips_df.to_csv(PATH_DIR_GTFS / "trips.txt", index=False)
print(f"Escrito: {PATH_DIR_GTFS / 'trips.txt'} ({len(trips_df)} viajes)")

Escrito: ../data/merida/gtfs-frequencies/trips.txt (56 viajes)
